In [2]:
# ==========================================================
# STUDENT 3
# NOTEBOOK 05
# ROBERTA + XGBOOST TRAINING
# ==========================================================

import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import joblib

from xgboost import XGBClassifier

from sklearn.metrics import accuracy_score

from sklearn.model_selection import RandomizedSearchCV

from scipy.stats import randint, uniform

warnings.filterwarnings("ignore")

In [3]:
# ==========================================================
# CONFIGURATION
# ==========================================================

NOTEBOOK_DIR = Path.cwd()

PROJECT_ROOT = NOTEBOOK_DIR.parent.parent

EMBEDDINGS = PROJECT_ROOT / "embeddings"

MODELS = PROJECT_ROOT / "models"

REPORTS = PROJECT_ROOT / "reports"

ROBERTA_EMBEDDINGS = EMBEDDINGS / "roberta"

print("="*70)
print("ROBERTA + XGBOOST TRAINING")
print("="*70)

ROBERTA + XGBOOST TRAINING


In [4]:
# ==========================================================
# LOAD ROBERTA EMBEDDINGS
# ==========================================================

X_train = np.load(
    ROBERTA_EMBEDDINGS / "train_embeddings.npy"
)

X_validation = np.load(
    ROBERTA_EMBEDDINGS / "validation_embeddings.npy"
)

y_train = np.load(
    ROBERTA_EMBEDDINGS / "train_labels.npy"
)

y_validation = np.load(
    ROBERTA_EMBEDDINGS / "validation_labels.npy"
)

print("="*70)
print("ROBERTA EMBEDDINGS LOADED")
print("="*70)

print("Training :", X_train.shape)
print("Validation :", X_validation.shape)

print("Training Labels :", y_train.shape)
print("Validation Labels :", y_validation.shape)

ROBERTA EMBEDDINGS LOADED
Training : (5000, 768)
Validation : (1000, 768)
Training Labels : (5000,)
Validation Labels : (1000,)


In [5]:
print("="*70)
print("TRAIN LABEL DISTRIBUTION")
print("="*70)

print(pd.Series(y_train).value_counts())

print()

print("="*70)
print("VALIDATION LABEL DISTRIBUTION")
print("="*70)

print(pd.Series(y_validation).value_counts())

TRAIN LABEL DISTRIBUTION
0    4552
1     448
Name: count, dtype: int64

VALIDATION LABEL DISTRIBUTION
0    910
1     90
Name: count, dtype: int64


In [6]:
baseline_model = XGBClassifier(

    objective="binary:logistic",

    eval_metric="logloss",

    n_estimators=300,

    max_depth=6,

    learning_rate=0.05,

    subsample=0.8,

    colsample_bytree=0.8,

    random_state=42,

    n_jobs=-1

)

print("="*70)
print("TRAINING BASELINE ROBERTA-XGBOOST")
print("="*70)

baseline_model.fit(
    X_train,
    y_train
)

print("="*70)
print("BASELINE TRAINING COMPLETED")
print("="*70)

TRAINING BASELINE ROBERTA-XGBOOST
BASELINE TRAINING COMPLETED


In [7]:
validation_predictions = baseline_model.predict(
    X_validation
)

accuracy = accuracy_score(
    y_validation,
    validation_predictions
)

print("="*70)
print("BASELINE VALIDATION ACCURACY")
print("="*70)

print(f"{accuracy:.4f}")

BASELINE VALIDATION ACCURACY
0.9900


In [8]:
parameter_space = {

    "n_estimators": randint(200,500),

    "max_depth": randint(3,8),

    "learning_rate": uniform(0.02,0.18),

    "subsample": uniform(0.70,0.30),

    "colsample_bytree": uniform(0.70,0.30),

    "min_child_weight": randint(1,6),

    "gamma": uniform(0.0,0.4)

}

In [9]:
random_search = RandomizedSearchCV(

    estimator=XGBClassifier(

        objective="binary:logistic",

        eval_metric="logloss",

        random_state=42,

        n_jobs=-1

    ),

    param_distributions=parameter_space,

    n_iter=30,

    scoring="f1",

    cv=3,

    random_state=42,

    verbose=2,

    n_jobs=-1,

    refit=True

)

In [11]:
print("="*70)
print("STARTING ROBERTA-XGBOOST OPTIMIZATION")
print("="*70)

random_search.fit(
    X_train,
    y_train
)

print("="*70)
print("OPTIMIZATION FINISHED")
print("="*70)

print(random_search.best_params_)

print()

print(random_search.best_score_)

STARTING ROBERTA-XGBOOST OPTIMIZATION
Fitting 3 folds for each of 30 candidates, totalling 90 fits
OPTIMIZATION FINISHED
{'colsample_bytree': np.float64(0.9139734361668984), 'gamma': np.float64(0.304314019446759), 'learning_rate': np.float64(0.12102989556250932), 'max_depth': 5, 'min_child_weight': 1, 'n_estimators': 406, 'subsample': np.float64(0.8282623055075649)}

0.965169182560487


In [12]:
best_xgb = random_search.best_estimator_

joblib.dump(

    best_xgb,

    MODELS /

    "best_roberta_xgboost.pkl"

)

print("="*70)
print("BEST ROBERTA-XGBOOST MODEL SAVED")
print("="*70)

BEST ROBERTA-XGBOOST MODEL SAVED


In [13]:
best_parameters = pd.DataFrame(
    [random_search.best_params_]
)

best_parameters["Best_F1"] = random_search.best_score_

display(best_parameters)

best_parameters.to_csv(

    REPORTS /

    "roberta_xgboost_best_parameters.csv",

    index=False

)

print("="*70)
print("BEST PARAMETERS SAVED")
print("="*70)

,colsample_bytree,gamma,learning_rate,max_depth,min_child_weight,n_estimators,subsample,Best_F1
0,0.913973,0.304314,0.12103,5,1,406,0.828262,0.965169


BEST PARAMETERS SAVED
